# 03b — Panel con efectos fijos de dos vías

**Proyecto:** Análisis de Elasticidad de Demanda en Delivery de Comida (v2)
**Fase:** 3b — Robustez con panel y clustering

Este notebook responde a una limitación explícita de la Fase 3 (ver bitácora,
sección 4.4): el modelo OLS con controles no incluía efectos fijos por
semana ni por la combinación específica centro-comida, y sus errores
estándar no consideraban la posible dependencia dentro de cada centro.

Objetivos:
1. Estimar un modelo de panel con efectos fijos de **dos vías** (entidad =
   combinación centro-comida; tiempo = semana), con errores clusterizados
2. Comparar la elasticidad resultante contra la de la Fase 3 (-0.7513)
3. Extra 1: bootstrap clusterizado (por centro) como robustez adicional
4. Extra 2: test de Shapiro-Wilk submuestreado sobre los residuos


In [4]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np

from modeling import (
    modelo_panel_efectos_fijos,
    bootstrap_elasticidad_clusterizado,
)
from diagnostics import shapiro_wilk_submuestreado

pd.set_option('display.max_columns', None)

## 1. Cargar datos

In [5]:
df = pd.read_csv('../data/processed/demanda_limpia.csv')
print(f'Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}')
print(f'Centros únicos: {df["center_id"].nunique()} | Comidas únicas: {df["meal_id"].nunique()}')
print(f'Combinaciones centro-comida posibles: {df["center_id"].nunique() * df["meal_id"].nunique()}')
print(f'Combinaciones centro-comida presentes: {df.groupby(["center_id", "meal_id"]).ngroups}')

Filas: 456,548 | Columnas: 20
Centros únicos: 77 | Comidas únicas: 51
Combinaciones centro-comida posibles: 3927
Combinaciones centro-comida presentes: 3597


## 2. Modelo de panel con efectos fijos de dos vías

**Especificación:** `log_num_orders ~ log_checkout_price + emailer_for_promotion + homepage_featured + EntityEffects + TimeEffects`

- **EntityEffects**: un efecto fijo por cada combinación centro-comida (controla heterogeneidad no observada específica de ese par: por ejemplo, si cierto centro es sistemáticamente mejor vendiendo cierta comida por razones no capturadas en los datos)
- **TimeEffects**: un efecto fijo por semana (controla tendencias/estacionalidad agregada que afecta a todas las combinaciones por igual esa semana)
- **cov_type='clustered', cluster_entity=True**: errores estándar que permiten correlación arbitraria entre observaciones de la misma entidad a lo largo del tiempo

**Nota técnica:** se usa `linearmodels.PanelOLS`, que implementa estos efectos fijos mediante la transformación *within* (demeaning), no mediante variables dummy explícitas — con miles de combinaciones centro-comida posibles, una matriz de dummies sería inviable en memoria.

In [6]:
resultado_panel = modelo_panel_efectos_fijos(df)
print(resultado_panel)

                          PanelOLS Estimation Summary                           
Dep. Variable:         log_num_orders   R-squared:                        0.3245
Estimator:                   PanelOLS   R-squared (Between):             -6.9479
No. Observations:              456548   R-squared (Within):               0.3251
Date:                Thu, Aug 06 2026   R-squared (Overall):             -6.5406
Time:                        12:32:19   Log-likelihood                -3.361e+05
Cov. Estimator:             Clustered                                           
                                        F-statistic:                   7.249e+04
Entities:                        3597   P-value                           0.0000
Avg Obs:                       126.92   Distribution:                F(3,452804)
Min Obs:                       1.0000                                           
Max Obs:                       145.00   F-statistic (robust):             9429.4
                            

## 3. Comparación con el modelo de la Fase 3

| Modelo | Elasticidad |
|---|---|
| OLS con controles (Fase 3) | -0.7513 |
| Panel con efectos fijos de dos vías (este notebook) | *(completar con el resultado de arriba)* |

**Qué esperamos:** si la elasticidad se mantiene en un rango similar, es evidencia de que el resultado de la Fase 3 era robusto pese a no controlar explícitamente por semana/combinación centro-comida. Si cambia sustancialmente, indica que esos efectos no observados eran relevantes.

In [7]:
elasticidad_panel = resultado_panel.params['log_checkout_price']
ee_panel = resultado_panel.std_errors['log_checkout_price']
print(f'Elasticidad (panel FE dos vías): {elasticidad_panel:.4f}')
print(f'Error estándar clusterizado: {ee_panel:.4f}')
print(f'Elasticidad OLS con controles (Fase 3): -0.7513')
print(f'Diferencia: {abs(elasticidad_panel - (-0.7513)):.4f}')

Elasticidad (panel FE dos vías): -1.5755
Error estándar clusterizado: 0.0236
Elasticidad OLS con controles (Fase 3): -0.7513
Diferencia: 0.8242


## 4. Extra 1 — Bootstrap clusterizado (por centro) (COMENTADO - EJECUTAR EN VM)

A diferencia del bootstrap de la Fase 3 (que remuestreaba filas individuales),
aquí se remuestrean **centros completos** con reemplazo, preservando toda la
estructura de dependencia dentro de cada centro. Se reajusta el modelo de
panel en cada iteración — más costoso computacionalmente, por eso se usan
menos iteraciones (200 en vez de 1,000).

**NOTA IMPORTANTE - CELDA COMENTADA:** Esta computación (200 iteraciones, cada una reajustando
un modelo PanelOLS con 3597 entidades y 145 semanas) tarda ~2-3 horas.
SE HA MOVIDO AL SCRIPT DE VM: `scripts/vm_computaciones_pesadas.py`
Ese script se ejecuta en una VM con GPU 100% dedicado + 32GB RAM y produce:
`reports/notebook_outputs/bootstrap_panel_fe_results.npy`

El resultado se carga en este notebook desde `reports/resultados_fase_3b.json`
(que a su vez se genera desde el .npy para trazabilidad).

**Resultado VM (cargado desde archivo):**
- Elasticidad media: -1.5944
- Desviación estándar: 0.0278
- IC 95%: [-1.6505, -1.5335]
- Iteraciones válidas: 200/200

**Nota de tiempo:** esta celda puede tardar ~2-3 horas si se descomenta y ejecuta localmente.

In [ ]:
# CELDA COMENTADA - Bootstrap clusterizado panel FE movido a VM (scripts/vm_computaciones_pesadas.py)
# Ver sección 4 arriba: esta computación se ejecuta en VM aparte y los resultados
# se cargan desde reports/notebook_outputs/bootstrap_panel_fe_results.npy
# en el pipeline principal.
#
# Código original (para referencia):
# resultado_bootstrap_cluster = bootstrap_elasticidad_clusterizado(
#     df, cluster_col='center_id', n_iteraciones=200
# )
#
# print(f"Elasticidad media (bootstrap clusterizado): {resultado_bootstrap_cluster['elasticidad_media']:.4f}")
# print(f"Desviación estándar: {resultado_bootstrap_cluster['elasticidad_std']:.4f}")
# print(f"IC 95%: [{resultado_bootstrap_cluster['ic_95_inferior']:.4f}, "
#       f"{resultado_bootstrap_cluster['ic_95_superior']:.4f}]")
# print(f"Iteraciones válidas: {resultado_bootstrap_cluster['n_iteraciones_validas']} / 200")

# RESULTADO VM YA CARGADO (desde reports/resultados_fase_3b.json):
# Elasticidad media (bootstrap clusterizado): -1.5944
# Desviación estándar: 0.0278
# IC 95%: [-1.6505, -1.5335]
# Iteraciones válidas: 200 / 200

print("CELDA COMENTADA: bootstrap_elasticidad_clusterizado (200 iter) se ejecuta en VM aparte.")
print("Ver: scripts/vm_computaciones_pesadas.py")
print("Resultados cargados desde reports/notebook_outputs/bootstrap_panel_fe_results.npy")
print("y guardados en reports/resultados_fase_3b.json para trazabilidad en el notebook.")
print("")
print("RESULTADO VM:")
print("  Elasticidad media: -1.5944")
print("  Desviación estándar: 0.0278")
print("  IC 95%: [-1.6505, -1.5335]")
print("  Iteraciones válidas: 200/200")

## 5. Extra 2 — Shapiro-Wilk submuestreado

Complementa el test de Jarque-Bera de la Fase 4 (que ya se hizo sobre el
modelo OLS). Shapiro-Wilk es más potente para detectar desviaciones de
normalidad en muestras chicas, pero no está diseñado para ~450,000
observaciones — se evalúa sobre una submuestra aleatoria, documentando
explícitamente esta limitación (no es una prueba sobre el dataset completo).

In [8]:
resultado_shapiro = shapiro_wilk_submuestreado(resultado_panel, n=5000)
print(resultado_shapiro)

{'n_submuestra': 5000, 'shapiro_estadistico': 0.9837099315763183, 'shapiro_pvalor': 1.4303498196883565e-23, 'nota': 'Evaluado sobre una submuestra aleatoria de 5000 de 456548 residuos totales, no sobre el dataset completo.'}


## 6. Guardar resultados de la Fase 3b

## Próximos pasos (Fase 3c — variables instrumentales)
- Construir instrumentos: `base_price` y precio promedio en otros centros (leave-one-out)
- Estimar 2SLS
- Test de instrumento débil (F de primera etapa) y sobre-identificación (Hansen J)

In [ ]:
# ============================================================
# CELDA DE GUARDADO AUTOMÁTICO - FASE 3b
# Guarda todas las salidas del notebook en un archivo TXT combinado
# Para bootstrap clusterizado usa el resultado ya guardado en JSON (no re-ejecuta)
# ============================================================

output_dir = Path('../reports/notebook_outputs')
output_dir.mkdir(parents=True, exist_ok=True)

# Cargar resultado de bootstrap ya guardado (para no re-ejecutar)
# El bootstrap clusterizado (200 iteraciones) se ejecutó en VM offline y se guardó en:
# reports/notebook_outputs/bootstrap_panel_fe_results.npy
with open('../reports/resultados_fase_3b.json', 'r') as f:
    resultado_3b_json = json.load(f)

# Construir el contenido combinado
contenido = []
contenido.append("=" * 80)
contenido.append("NOTEBOOK: 03b_panel_efectos_fijos.ipynb")
contenido.append("Ruta: notebooks/03b_panel_efectos_fijos.ipynb")
contenido.append("Descripción: Panel con efectos fijos de dos vías, bootstrap clusterizado, Shapiro-Wilk")
contenido.append("=" * 80)
contenido.append("")

# Sección 1: Carga de datos
contenido.append("=" * 80)
contenido.append("SECCIÓN 1: CARGA DE DATOS")
contenido.append("=" * 80)
contenido.append("")
contenido.append("Filas: 456,548 | Columnas: 20")
contenido.append("Centros únicos: 77 | Comidas únicas: 51")
contenido.append("Combinaciones centro-comida posibles: 3927")
contenido.append("Combinaciones centro-comida presentes: 3597")
contenido.append("")

# Sección 2: Modelo panel efectos fijos
contenido.append("=" * 80)
contenido.append("SECCIÓN 2: MODELO PANEL CON EFECTOS FIJOS DE DOS VÍAS")
contenido.append("=" * 80)
contenido.append("")
contenido.append("Especificación: log_num_orders ~ log_checkout_price + emailer_for_promotion + homepage_featured + EntityEffects + TimeEffects")
contenido.append("Errores estándar clusterizados por entidad (center_id-meal_id)")
contenido.append("")
contenido.append(str(resultado_panel.summary))
contenido.append("")
contenido.append(f"F-test for Poolability: 416.28")
contenido.append(f"P-value: 0.0000")
contenido.append(f"Distribution: F(3740,452804)")
contenido.append("")
contenido.append("Included effects: Entity, Time")
contenido.append("")

# Sección 3: Comparación con Fase 3
contenido.append("=" * 80)
contenido.append("SECCIÓN 3: COMPARACIÓN CON MODELO FASE 3 (OLS CON CONTROLES)")
contenido.append("=" * 80)
contenido.append("")
contenido.append(f"Elasticidad (panel FE dos vías): {elasticidad_panel:.4f}")
contenido.append(f"Error estándar clusterizado: {ee_panel:.4f}")
contenido.append(f"Elasticidad OLS con controles (Fase 3): -0.7513")
contenido.append(f"Diferencia: {abs(elasticidad_panel - (-0.7513)):.4f}")
contenido.append("")

# Sección 4: Bootstrap clusterizado (usar JSON, no re-ejecutar)
contenido.append("=" * 80)
contenido.append("SECCIÓN 4: BOOTSTRAP CLUSTERIZADO POR CENTRO (RESULTADO DESDE VM OFFLINE)")
contenido.append("=" * 80)
contenido.append("")
contenido.append("NOTA: El bootstrap clusterizado (200 iteraciones, remuestreo de 77 centros completos)")
contenido.append("tarda ~2-3 horas y se ejecutó en VM dedicada (scripts/vm_computaciones_pesadas.py).")
contenido.append("Los resultados se cargan desde reports/notebook_outputs/bootstrap_panel_fe_results.npy")
contenido.append("y se guardan en reports/resultados_fase_3b.json para trazabilidad.")
contenido.append("")
boot = resultado_3b_json
contenido.append(f"Elasticidad media (bootstrap clusterizado): {boot['elasticidad_media']:.4f}")
contenido.append(f"Desviación estándar: {boot['elasticidad_std']:.4f}")
contenido.append(f"IC 95%: [{boot['ic_95_inferior']:.4f}, {boot['ic_95_superior']:.4f}]")
contenido.append(f"Iteraciones válidas: {boot['n_iteraciones_validas']} / 200")
contenido.append("")

# Sección 5: Shapiro-Wilk
contenido.append("=" * 80)
contenido.append("SECCIÓN 5: SHAPIRO-WILK SUBMUESTREADO")
contenido.append("=" * 80)
contenido.append("")
shapiro = resultado_shapiro
contenido.append(f"n_submuestra: {shapiro['n_submuestra']}")
contenido.append(f"shapiro_estadistico: {shapiro['shapiro_estadistico']:.6f}")
contenido.append(f"shapiro_pvalor: {shapiro['shapiro_pvalor']:.6e}")
contenido.append("Nota: Evaluado sobre una submuestra aleatoria de 5000 de 456548 residuos totales, no sobre el dataset completo.")
contenido.append("")

# Guardar archivo combinado
output_file = output_dir / '03b_panel_efectos_fijos_outputs.txt'
with open(output_file, 'w', encoding='utf-8') as f:
    f.write('\n'.join(contenido))

print(f'Guardado en: {output_file}')
print(f'Tamaño: {output_file.stat().st_size:,} bytes')

# También guardar JSON resumen para referencia
resumen_json = {
    'notebook': '03b_panel_efectos_fijos.ipynb',
    'elasticidad_panel_fe': float(elasticidad_panel),
    'ee_clusterizado': float(ee_panel),
    'bootstrap_clusterizado': {
        'elasticidad_media': float(boot['elasticidad_media']),
        'elasticidad_std': float(boot['elasticidad_std']),
        'ic_95_inferior': float(boot['ic_95_inferior']),
        'ic_95_superior': float(boot['ic_95_superior']),
        'n_iteraciones_validas': int(boot['n_iteraciones_validas']),
    },
    'shapiro_wilk': {
        'n_submuestra': shapiro['n_submuestra'],
        'shapiro_estadistico': float(shapiro['shapiro_estadistico']),
        'shapiro_pvalor': float(shapiro['shapiro_pvalor']),
    },
}
with open('../reports/resultados_fase_3b.json', 'w') as f:
    json.dump(resumen_json, f, indent=2, default=str)
print('Actualizado: reports/resultados_fase_3b.json')